# Homework 4

This file contains code to solve problems on Homework 8 assigned by Akil Narayan. The code written here was generated by ChatGPT.

### Problem 5.5

The below description was copied and pasted from the homework assignment.

In [2]:
import numpy as np
from time import perf_counter

First define some functions we will use throughout the two methods.

In [3]:
def random_complex_unit_vector(n, rng):
    """Return a random unit vector in C^n."""
    x = rng.standard_normal(n) + 1j * rng.standard_normal(n)
    return x / np.linalg.norm(x)


def relative_residual(A, lam, v):
    """
    ||Av - lambda*v||_2 / (max(1, |lambda|) ||v||_2).
    """
    return np.linalg.norm(A @ v - lam * v) / (
        max(1.0, abs(lam)) * np.linalg.norm(v)
    )


def greedy_spectral_error(computed, oracle):
    """
    Permutation-invariant relative error, using the greedy matching
    procedure suggested in the problem.
    """
    computed = np.array(computed, dtype=complex)
    remaining_oracle = list(np.array(oracle, dtype=complex))

    computed = computed[np.argsort(np.abs(computed))]
    squared_error = 0.0

    for lam_hat in computed:
        distances = [abs(lam_hat - lam) for lam in remaining_oracle]
        j = int(np.argmin(distances))
        squared_error += distances[j] ** 2
        remaining_oracle.pop(j)

    denominator = max(1.0, np.linalg.norm(oracle))
    return np.sqrt(squared_error) / denominator

Define RQI method

In [4]:
def rqi_one_eigenpair(A, tol=1e-10, maxit=1000, rng=None):
    """
    Find one approximate eigenpair of A using Rayleigh quotient iteration.

    Returns:
        lam      approximate eigenvalue
        v        approximate unit eigenvector
        iters    number of RQI iterations used
    """
    n = A.shape[0]

    if rng is None:
        rng = np.random.default_rng()

    v = random_complex_unit_vector(n, rng)

    for k in range(1, maxit + 1):
        # Rayleigh quotient: v^* A v because ||v||_2 = 1
        lam = np.vdot(v, A @ v)

        if relative_residual(A, lam, v) <= tol:
            return lam, v, k

        # Inverse iteration with the Rayleigh quotient as the shift
        try:
            y = np.linalg.solve(A - lam * np.eye(n, dtype=complex), v)
        except np.linalg.LinAlgError:
            # If the shifted matrix is numerically singular, restart.
            v = random_complex_unit_vector(n, rng)
            continue

        y_norm = np.linalg.norm(y)
        if y_norm == 0:
            v = random_complex_unit_vector(n, rng)
        else:
            v = y / y_norm

    raise RuntimeError("RQI did not converge within maxit iterations.")


def full_spectrum_rqi(A, tol=1e-10, maxit=1000, max_restarts=20, seed=0):
    """
    Find all eigenvalues via repeated RQI and Hotelling deflation.

    Returns:
        eigenvalues
        total_iterations
    """
    A_work = np.array(A, dtype=complex, copy=True)
    n = A.shape[0]
    rng = np.random.default_rng(seed)

    eigenvalues = []
    total_iterations = 0

    for j in range(n):
        converged = False

        for _ in range(max_restarts):
            try:
                lam, v, iters = rqi_one_eigenpair(
                    A_work, tol=tol, maxit=maxit, rng=rng
                )
                converged = True
                break
            except RuntimeError:
                pass

        if not converged:
            raise RuntimeError(f"RQI failed while computing eigenvalue {j + 1}.")

        eigenvalues.append(lam)
        total_iterations += iters

        # Hotelling deflation:
        # A <- A - lambda * (v v^*) / (v^*v)
        A_work = A_work - lam * np.outer(v, v.conj()) / np.vdot(v, v)

    return np.array(eigenvalues), total_iterations

Define QR shifted method

In [5]:
def full_spectrum_shifted_qr(A, tol=1e-10, maxit=200000):
    """
    Shifted QR iteration with Rayleigh quotient shifts.

    Deflation occurs when the off-diagonal part of the final row of
    the active matrix is small:
        ||A_k[m-1, :m-1]||_2 <= tol.

    Returns:
        eigenvalues
        number of QR iterations
    """
    H = np.array(A, dtype=complex, copy=True)
    n = H.shape[0]

    eigenvalues = []
    total_iterations = 0
    m = n

    while m > 1:
        # Deflate the last eigenvalue of the active m x m block.
        if np.linalg.norm(H[m - 1, :m - 1]) <= tol:
            eigenvalues.append(H[m - 1, m - 1])
            m -= 1
            continue

        if total_iterations >= maxit:
            raise RuntimeError("QR iteration did not converge within maxit iterations.")

        active_block = H[:m, :m]

        # Rayleigh quotient shift using e_m:
        # e_m^* A e_m = A[m-1, m-1]
        mu = active_block[m - 1, m - 1]

        Q, R = np.linalg.qr(active_block - mu * np.eye(m, dtype=complex))
        H[:m, :m] = R @ Q + mu * np.eye(m, dtype=complex)

        total_iterations += 1

    eigenvalues.append(H[0, 0])

    return np.array(eigenvalues), total_iterations

Define the matrices requested in the problem